# Day 45 · Build, tune or retrieve? A cost and latency calculator

**First-principles question:** For Acme's volume, which design is cheapest at the quality they need, and how fast is it?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import numpy as np
import pandas as pd
from lab.acme import load_results

## 1 · What your versions really use

In [ ]:
rows = []
for version in ["A", "B", "B_retrieved", "C", "C-managed"]:
    try:
        r = load_results(version)
    except FileNotFoundError:
        continue
    rows.append({"version": version, "tokens_in": np.mean([x.get("tokens_in", 0) for x in r]),
                 "tokens_out": np.mean([x.get("tokens_out", 0) for x in r]),
                 "p95 s": float(np.percentile([x.get("seconds", 0) for x in r], 95))})
measured = pd.DataFrame(rows).set_index("version").round(1)
measured

## 2 · The calculator

In [ ]:
def monthly(emails, tokens_in, tokens_out, price_in, price_out, ttft_s, per_token_s, fixed_inr=0.0):
    """₹ per month and the wait for one reply. Prices are ₹ per million tokens."""
    # TODO: cost = emails * (tokens_in * price_in + tokens_out * price_out) / 1e6 + fixed_inr
    #       wait = ttft_s + tokens_out * per_token_s
    ...

OPTIONS = {
    # TODO: fill in from your measurements and a provider's price list
    "prompt only": dict(tokens_in=2500, tokens_out=200, price_in=40, price_out=160, ttft_s=0.6, per_token_s=0.02),
    "hybrid RAG": dict(tokens_in=1500, tokens_out=200, price_in=40, price_out=160, ttft_s=0.5, per_token_s=0.02, fixed_inr=4000),
    "fine-tuned small model": dict(tokens_in=600, tokens_out=200, price_in=15, price_out=60, ttft_s=0.3, per_token_s=0.01, fixed_inr=15000),
}
table = pd.DataFrame({name: {f"{v:,} emails": "₹{:,.0f} · {:.1f} s".format(*monthly(v, **o)) for v in [10_000, 100_000, 1_000_000]}
                      for name, o in OPTIONS.items()}).T
table

**The break-even point, and the one sentence for Acme's CFO:** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
cost, wait = monthly(1_000_000, 1000, 100, 10, 20, 0.5, 0.01)
assert abs(cost - 12_000) < 1e-6, "cost = emails × (in × price_in + out × price_out) ÷ 1e6"
assert abs(wait - 1.5) < 1e-9, "wait = TTFT + tokens out × time per token"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")